In [1]:
import os
import json

games = []

for file in os.listdir("games"):
    if file.endswith(".json"):
        with open(os.path.join("games", file)) as f:
            games.append(json.load(f))

games

[{'title': 'FIFA 21',
  'description': 'A football simulation video game developed by EA Sports.',
  'release_date': '2020-10-09',
  'platforms': ['PS4', 'Xbox One', 'PC'],
  'publisher': 'Electronic Arts'},
 {'title': 'God of War Ragnarok',
  'description': 'An action-adventure game developed by Santa Monica Studio.',
  'release_date': '2022-11-09',
  'platforms': ['PS4', 'PS5'],
  'publisher': 'Sony Interactive Entertainment'},
 {'title': "PlayerUnknown's Battlegrounds",
  'description': 'A battle royale game developed by PUBG Studios where players compete to be the last survivor.',
  'release_date': '2017-12-20',
  'platforms': ['PC', 'Xbox One', 'PS4', 'Mobile'],
  'publisher': 'PUBG Corporation'},
 {'title': 'Road Rash',
  'description': 'A motorcycle racing and combat video game developed by Electronic Arts.',
  'release_date': '1991-09-01',
  'platforms': ['Sega Genesis', 'PlayStation', 'PC'],
  'publisher': 'Electronic Arts'},
 {'title': 'Tekken 7',
  'description': 'A fighting

In [9]:
import chromadb
from chromadb.utils import embedding_functions
import json

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.Client()

# reset collection
try:
    client.delete_collection("games")
except:
    pass

collection = client.create_collection(
    name="games",
    embedding_function=embedding_function
)

for game in games:

    doc_text = f"""
    Title: {game.get('title','')}
    Developer: {game.get('developer','')}
    Publisher: {game.get('publisher','')}
    Genre: {game.get('genre','')}
    Release Date: {game.get('release_date','')}
    Platforms: {game.get('platforms','')}
    """

    collection.add(
        documents=[doc_text],
        ids=[game.get("title","unknown")],
        metadatas=[game]
    )

    print("Games successfully stored with embeddings.")
    collection.add(
        documents=[doc_text],
        ids=[game["title"]],
        metadatas=[game]
    )

    print("Games successfully stored with MiniLM embeddings.")

Games successfully stored with embeddings.
Games successfully stored with MiniLM embeddings.
Games successfully stored with embeddings.
Games successfully stored with MiniLM embeddings.
Games successfully stored with embeddings.
Games successfully stored with MiniLM embeddings.
Games successfully stored with embeddings.
Games successfully stored with MiniLM embeddings.
Games successfully stored with embeddings.
Games successfully stored with MiniLM embeddings.


In [ ]:
import json

def retrieve_game_tool(query):

    results = collection.query(
        query_texts=[query],
        n_results=1
    )

    if not results:
        return None

    if not results.get("documents") or not results["documents"][0]:
        return None

    document = results["documents"][0][0]
    distance = results["distances"][0][0]

    try:
        game = json.loads(document)

        pretty_response = (
            f"🎮 Game Information\n\n"
            f"Title: {game.get('title', 'Unknown')}\n"
            f"Developer: {game.get('developer', 'Unknown')}\n"
            f"Publisher: {game.get('publisher', 'Unknown')}\n"
            f"Genre: {game.get('genre', 'Unknown')}\n"
            f"Release Date: {game.get('release_date', 'Unknown')}\n"
            f"Platforms: {', '.join(game.get('platforms', []))}\n\n"
            f"(Source: stored web knowledge)"
        )

    except Exception:
        pretty_response = (
            f"📄 Information\n\n"
            f"{document}\n\n"
            f"(Source: stored web knowledge)"
        )

    return {
        "response": pretty_response,
        "document": document,
        "distance": distance
    }

In [4]:
retrieve_game_tool("Which company made PUBG?")

{'response': "📄 Information\n\n\n    Title: PlayerUnknown's Battlegrounds\n    Developer: \n    Publisher: PUBG Corporation\n    Genre: \n    Release Date: 2017-12-20\n    Platforms: ['PC', 'Xbox One', 'PS4', 'Mobile']\n    \n\n(Source: stored web knowledge)",
 'document': "\n    Title: PlayerUnknown's Battlegrounds\n    Developer: \n    Publisher: PUBG Corporation\n    Genre: \n    Release Date: 2017-12-20\n    Platforms: ['PC', 'Xbox One', 'PS4', 'Mobile']\n    ",
 'distance': 0.5783842206001282}

In [ ]:
# from openai import OpenAI
# import os

# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# not using open AI, instead using ollama mistral for LLM.. it is opensource

In [14]:
# Testing ollama mistral installation here
import ollama

ollama.chat(
    model="mistral",
    messages=[{"role": "user", "content": "Say hello"}]
)

ChatResponse(model='mistral', created_at='2026-03-14T19:03:42.8105784Z', done=True, done_reason='stop', total_duration=3008532800, load_duration=67456000, prompt_eval_count=7, prompt_eval_duration=808945400, eval_count=11, eval_duration=2122514100, message=Message(role='assistant', content=' Hello there! How can I assist you today?', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [7]:
import ollama
import json

def evaluate_retrieval_tool(question, document):

    prompt = f"""
You are evaluating whether a retrieved document is useful to answer a user's question.
Consider developer/development and publisher as same thing.

Question:
{question}

Retrieved document:
{document}

Respond ONLY in JSON format:

{{
 "useful": true or false,
 "reason": "give reason if loaded document is relevant"
}}
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}]
    )

    content = response["message"]["content"]

    try:
        return json.loads(content)
    except:
        return {"useful": False, "reason": "Could not parse response"}



In [8]:
retrieval = retrieve_game_tool("Who developed god of war?")

evaluation = evaluate_retrieval_tool(
    "Who developed god of war?",
    retrieval["document"]
)

evaluation

{'useful': False,
 'reason': 'The document does not provide information about who developed God of War. It only mentions the publisher, Sony Interactive Entertainment, but not the developer.'}

In [15]:
from tavily import TavilyClient
import os

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def web_search_tool(query):
    """
    Tool: Perform web search when internal knowledge is insufficient
    """

    results = tavily.search(
        query=query,
        max_results=3
    )

    return results

In [16]:
web_search_tool("When was counter strike released?")

{'query': 'When was counter strike released?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Counter-Strike_(video_game)',
   'title': 'Counter-Strike (video game) - Wikipedia',
   'content': '# *Counter-Strike* (video game). ***Counter-Strike*** (also known as ***Half-Life: Counter-Strike*** or ***Counter-Strike 1.6***) is a 2000 tactical first-person shooter game developed by Valve Corporation and published by Sierra Studios. *Counter-Strike* is a team-based multiplayer first-person shooter video game in which players play as Terrorists (T) or Counter-Terrorists (CT). On April 12, 2000, Valve Software announced a partnership with the *Counter-Strike* Team, confirming that *Counter-Strike 1.0* would be included in an upcoming *Half-Life "Half-Life (video game)")* patch. It was launched under the name *Half-Life: Counter-Strike* because according to Jess Cliffe, the game did not have a strong identity. *Counter-Strike*

In [13]:
import json

class UdaPlayAgent:

    def __init__(self):
        self.history = []

    def run(self, question):

            print("\nQUESTION:", question)
            print("Conversation history:", self.history)

            # resolve simple follow-up questions
            if question.lower().startswith("when was it") and self.history:
                question = self.history[-1]

            self.history.append(question)

            state = "RETRIEVE"
            retrieval = None
            document = None

            while state != "END":

                # ----------------------------
                # STATE: RETRIEVE
                # ----------------------------
                if state == "RETRIEVE":

                    print("\nSTATE: RETRIEVE")

                    retrieval = retrieve_game_tool(question)

                    if retrieval is None:
                        print("No internal knowledge found.")
                        state = "WEB_SEARCH"
                    else:
                        document = retrieval["document"]
                        state = "EVALUATE"

                # ----------------------------
                # STATE: EVALUATE
                # ----------------------------
                elif state == "EVALUATE":

                    print("\nSTATE: EVALUATE")

                    evaluation = evaluate_retrieval_tool(question, document)

                    print("Evaluation:", evaluation)

                    if evaluation.get("useful"):
                        state = "ANSWER"
                    else:
                        state = "WEB_SEARCH"

                # ----------------------------
                # STATE: ANSWER
                # ----------------------------
                elif state == "ANSWER":

                    print("\nSTATE: ANSWER FROM INTERNAL KNOWLEDGE")

                    try:
                        game = json.loads(document)

                        answer = f"""
        Game: {game.get('title')}
        Release Date: {game.get('release_date')}
        Publisher: {game.get('publisher')}
        Platforms: {', '.join(game.get('platforms', []))}
        """

                        state = "END"
                        return answer

                    except:

                        return f"""
        Information:

        {document}

        (Source: stored web knowledge)
        """

                # ----------------------------
                # STATE: WEB SEARCH
                # ----------------------------
                elif state == "WEB_SEARCH":

                    print("\nSTATE: WEB SEARCH")

                    web_results = web_search_tool(question)

                    if not web_results:
                        return "No web results found."

                    results = web_results.get("results", [])

                    if not results:
                        return "No relevant web results found."

                    result = results[0]

                    content = result.get("content", "No content available.")
                    source = result.get("url", "Unknown source")

                    #store_web_knowledge(content, source)

                    state = "END"

                    return f"{content}\n\nSource: {source}"

In [55]:
agent = UdaPlayAgent()

In [56]:
agent.run("Who developed FIFA 21?")


QUESTION: Who developed FIFA 21?
Conversation history: []

STATE: RETRIEVE

STATE: EVALUATE
Evaluation: {'useful': True, 'reason': "The document lists 'Publisher: Electronic Arts', which is the same as 'Developer' in this context, and Electronic Arts is the developer of FIFA 21."}

STATE: ANSWER FROM INTERNAL KNOWLEDGE


"\n        Information:\n\n        \n    Title: FIFA 21\n    Developer: \n    Publisher: Electronic Arts\n    Genre: \n    Release Date: 2020-10-09\n    Platforms: ['PS4', 'Xbox One', 'PC']\n    \n\n        (Source: stored web knowledge)\n        "

In [57]:
agent.run("When was Tekken 7 released?")


QUESTION: When was Tekken 7 released?
Conversation history: ['Who developed FIFA 21?']

STATE: RETRIEVE

STATE: EVALUATE
Evaluation: {'useful': True, 'reason': "The document provides the release date of Tekken 7, which is the information required to answer the user's question."}

STATE: ANSWER FROM INTERNAL KNOWLEDGE


"\n        Information:\n\n        \n    Title: Tekken 7\n    Developer: \n    Publisher: Bandai Namco Entertainment\n    Genre: \n    Release Date: 2017-06-02\n    Platforms: ['PS4', 'Xbox One', 'PC']\n    \n\n        (Source: stored web knowledge)\n        "

In [58]:
agent.run("When is GTA 6 released?")


QUESTION: When is GTA 6 released?
Conversation history: ['Who developed FIFA 21?', 'When was Tekken 7 released?']

STATE: RETRIEVE

STATE: EVALUATE
Evaluation: {'useful': False, 'reason': 'The document does not provide information about the release date of GTA 6, as it pertains to Tekken 7 by Bandai Namco Entertainment.'}

STATE: WEB SEARCH


'“Grand Theft Auto VI” has been delayed again by Take-Two Interactive’s Rockstar Games, this time pushed six months from its previously set May 2026 release date. “In this instance, of course, we’re seeking to release the most extraordinary title anyone’s ever seen in the history of entertainment,” Zelnick told *Variety* in an interview ahead of the announcement Thursday. And in this instance, Rockstar Games believes a limited amount of additional time is required for polish to support that view.”. The Take-Two team says that the recent firings of approximately 30-40 employees at Rockstar Games in the U.K. is unrelated to the “GTA 6” delay. Now in development for more than a decade, Rockstar’s “GTA 6” was originally on target for a fall 2025 release until the developer finally announced this May that the game would not be ready until May 26, 2026. In addition to the “GTA 6” shuffle news, Take-Two updated its release calendar Thursday to include “BioShock next iteration” from 2K and “To

In [ ]:
# Not using this for now
import uuid

def store_web_knowledge(text, source):

    collection.add(
        documents=[text],
        ids=[str(uuid.uuid4())],
        metadatas=[{"source": source}]
    )

    print("New knowledge stored in vector database.")

In [17]:
agent = UdaPlayAgent()
agent.run("When was Cricket 19 released adn who was the publisher?")


QUESTION: When was Cricket 19 released adn who was the publisher?
Conversation history: []

STATE: RETRIEVE

STATE: EVALUATE
Evaluation: {'useful': False, 'reason': 'The document is not relevant to the question about the release date and publisher of Cricket 19. The document describes FIFA 21, not Cricket 19.'}

STATE: WEB SEARCH


'The game was published by Big Ant Studios on May 28, 2019. Released on. Nintendo. PlayStation. PC. Xbox. Release dates. May 28, 2019 -\n\nSource: https://newzoo.com/games/cricket-19'